# Egg_Producing_Chickens_Exploration.ipynb
**ICS 3202 — Artificial Intelligence | Project Deliverable 1**

**Project:** An Integrated Enterprise Resource Planning System for Poultry Farm Management

**Dataset:** [Egg Producing Chickens](https://www.kaggle.com/datasets/phuzoman/egg-producing-chickens) (Kaggle, `phuzoman/egg-producing-chickens`)
1,000 daily observations across chicken breeds, recording physical attributes, feed intake, sunlight exposure, and eggs laid per day. Note: partially artificially generated from domain knowledge rather than raw farm sensor data — suitable for this exploration exercise, worth supplementing with real farm data later in the project.

**Expected output variable of the final application:** `EggsPerDay` — the number of eggs a bird is expected to lay on a given day, predicted from feed intake, age, weight, breed, and sunlight exposure. This feeds the ERP's **Production Tracking** module (forecast expected yield vs. actual, flag underperforming flocks) and **Feed Management** module (relate feed spend to output).

## Dataset Discovery (Instructions 1 & 2)

Open-source datasets considered as relevant to the poultry ERP's ML engine, and the output variable each would support:

| Dataset | Source | Candidate output variable |
|---|---|---|
| **Egg Producing Chickens** (selected) | Kaggle: `phuzoman/egg-producing-chickens` | `EggsPerDay` — daily egg yield per bird |
| Prediction of Egg Production Rate in Poultry | Kaggle: `deepikabidri/prediction-of-egg-production-rate-in-poultry` | egg production rate, from humidex/air/water quality |
| Environmental Effect on Egg Production | Kaggle: `faysal1998/environmental-effect-on-egg-production` | egg production rate, from environmental conditions |
| Poultry Farm Management Dataset (Sabarhi Hatcheries) | IEEE DataPort | weekly mortality rate / cull rate, from feed & medicine records |
| Eggs and Butter (USDA-style historical) | Kaggle: `datasciencedonut/eggs-and-butter` | national egg production trend (too aggregate for a single-farm ERP) |

**Why `Egg Producing Chickens` was selected:** it is the only candidate that is (a) a clean single CSV rather than aggregate/national statistics, (b) has an output variable (`EggsPerDay`) that maps directly onto the ERP's own **Production Tracking** and **Feed Management** modules, and (c) includes enough predictor variables (feed intake, age, weight, breed, sunlight exposure) to support a genuine regression model rather than a toy example. Its main limitation, addressed below, is that it is partly synthetically generated.

## Setup — download the dataset

In [ ]:
# Run this in Google Colab.
# 1) Upload your kaggle.json (Kaggle account -> Settings -> Create New API Token)
from google.colab import files
uploaded = files.upload()  # select kaggle.json when prompted

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!pip install -q kaggle
!kaggle datasets download -d phuzoman/egg-producing-chickens
!unzip -q -o egg-producing-chickens.zip -d egg_data

In [ ]:
import pandas as pd
import numpy as np
import glob

# The dataset ships as a single CSV
csv_path = glob.glob('egg_data/*.csv')[0]
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Resolve column names defensively: the CSV's exact spelling/casing is confirmed at
# runtime rather than assumed, so a renamed column gives a clear message, not a KeyError.
print("Columns in dataset:", list(df.columns))

def find_col(name):
    """Return the actual column matching `name` (case/underscore-insensitive), or None."""
    want = name.lower().replace("_", "").replace(" ", "")
    for c in df.columns:
        if c.lower().replace("_", "").replace(" ", "") == want:
            return c
    return None

## a. How many rows and columns are contained in the dataset? (3 marks)

In [ ]:
num_rows, num_cols = df.shape
print(f"Rows: {num_rows}")
print(f"Columns: {num_cols}")

## b. What datatypes are contained in the dataset? (3 marks)

In [ ]:
df.dtypes

## c. Is the dataset complete? i.e. no missing values? (2 marks)

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)
print()
print("Dataset is complete (no missing values):", df.isnull().sum().sum() == 0)

## d. Slice out the first 15 rows and last 20 rows, merge into `df_sample`, display it (4 marks)

In [ ]:
first_15 = df.head(15)
last_20 = df.tail(20)

df_sample = pd.concat([first_15, last_20], ignore_index=True)
df_sample

## Notes / next steps
- `EggsPerDay` is the target for a regression model (predict expected daily yield). `AmountOfFeed`, `Age`, `GallusWeight`, and `SunLightExposure` are the strongest likely predictors based on the domain description.
- Categorical fields (breed, comb type, colors, plumage) will need encoding (one-hot / ordinal) before modeling.
- Because this dataset is synthetic, treat any accuracy numbers from it as a proof-of-concept for the ERP's forecasting module, not a production-ready model — real feed/production logs from the Nairobi-area farms in the study should replace or augment it before deployment.

---
## Additional exploration (beyond the required questions)
Not part of the graded a–d questions, but strengthens the case for this dataset feeding the ML engine.

### Duplicate rows check

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

### Summary statistics for numeric columns

In [ ]:
df.describe()

### Relationship between feed intake, age, and egg output

In [ ]:
import matplotlib.pyplot as plt

target = find_col('EggsPerDay')
predictors = [c for c in (find_col(n) for n in
              ['Age', 'GallusWeight', 'AmountOfFeed', 'SunLightExposure']) if c]

if target is None:
    print("Target column not found. Numeric columns available:",
          df.select_dtypes(include='number').columns.tolist())
else:
    numeric_cols = [c for c in predictors + [target]
                    if pd.api.types.is_numeric_dtype(df[c])]
    corr = df[numeric_cols].corr()
    print(corr[target].sort_values(ascending=False))

    to_plot = [c for c in numeric_cols if c != target][:2]
    if to_plot:
        fig, axes = plt.subplots(1, len(to_plot),
                                 figsize=(6 * len(to_plot), 4), squeeze=False)
        for ax, col in zip(axes[0], to_plot):
            ax.scatter(df[col], df[target], alpha=0.4)
            ax.set_xlabel(col)
            ax.set_ylabel(target)
            ax.set_title(f'{col} vs. {target}')
        plt.tight_layout()
        plt.show()

### Breed distribution

In [ ]:
breed_col = find_col('GallusBreed') or find_col('Breed')

if breed_col:
    display(df[breed_col].value_counts())
else:
    print("No breed column found. Categorical columns available:")
    print(df.select_dtypes(include='object').columns.tolist())